# 🏪 Calculator Profit Afacere — Ghid de Utilizare

**Romania 2026 | Chioșc / Container | Versiunea 1.0**

---

Acest notebook explică:
1. Cum să instalezi dependențele
2. Cum funcționează calculatorul (CONFIG + funcții)
3. Cum să modifici valorile implicite
4. Cum să pornești aplicația Streamlit
5. Cum să interpretezi rezultatele

## Pasul 1 — Instalare dependențe

Rulează celula de mai jos o singură dată pentru a instala toate pachetele necesare.

In [ ]:
# Instalare pachete necesare
import subprocess
result = subprocess.run(
    ['pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
print(result.stdout or 'Pachetele sunt deja instalate ✅')
if result.returncode != 0:
    print('EROARE:', result.stderr)

## Pasul 2 — Regulile fiscale (CONFIG)

Toate cotele fiscale sunt centralizate în obiectul `CONFIG` din `app.py`. **Nu trebuie să modifici nimic altundeva** — dacă se schimbă legislația, editezi doar CONFIG.

| Parametru | Valoare | Descriere |
|---|---|---|
| `COTA_IMPOZIT_PROFIT` | 16% | Impozit profit SRL standard |
| `COTA_IMPOZIT_DIVIDENDE` | 16% | Impozit pe dividende |
| `COTA_CASS_DIVIDENDE` | 10% | CASS aplicat pe dividende (din iulie 2026) |
| `CASS_PLAFON_6SAL` | 25.950 RON | Plafon 1: 6 × salariu minim |
| `CASS_PLAFON_12SAL` | 51.900 RON | Plafon 2: 12 × salariu minim |
| `CASS_PLAFON_24SAL` | 103.800 RON | Plafon 3: 24 × salariu minim |
| `CAS_ANGAJAT` | 25% | Contribuție pensie angajat |
| `CASS_ANGAJAT` | 10% | Asigurare sănătate angajat |
| `IMPOZIT_VENIT` | 10% | Impozit venit din salariu |
| `CAM_ANGAJATOR` | 2,25% | Contribuție asiguratorie muncă (angajator) |
| `SALARIU_MINIM_BRUT` | 4.325 RON | Salariu minim brut (după iulie 2026) |

**Formula salarizare:**
```
Net = Brut × 0.585
Brut = Net / 0.585
Cost angajator = Brut × 1.0225
```

In [ ]:
# Vizualizare CONFIG și calcule de probă
import sys
import os
sys.path.insert(0, os.getcwd())

CONFIG = {
    'COTA_IMPOZIT_PROFIT': 0.16,
    'COTA_IMPOZIT_DIVIDENDE': 0.16,
    'COTA_CASS_DIVIDENDE': 0.10,
    'CASS_PLAFON_6SAL': 25950,
    'CASS_PLAFON_12SAL': 51900,
    'CASS_PLAFON_24SAL': 103800,
    'CAS_ANGAJAT': 0.25,
    'CASS_ANGAJAT': 0.10,
    'IMPOZIT_VENIT': 0.10,
    'CAM_ANGAJATOR': 0.0225,
    'SALARIU_MINIM_BRUT': 4325,
    'CURS_EUR_RON_DEFAULT': 5.10,
}

# Exemplu: salariu net dorit = 6.500 RON
net_dorit = 6500
brut = net_dorit / 0.585
cost_angajator = brut * (1 + CONFIG['CAM_ANGAJATOR'])

print(f"Salariu net dorit    : {net_dorit:,.0f} RON")
print(f"Salariu brut necesar : {brut:,.2f} RON")
print(f"CAS angajat (25%)    : {brut * 0.25:,.2f} RON")
print(f"CASS angajat (10%)   : {brut * 0.10:,.2f} RON")
print(f"Impozit venit (10%)  : {(brut * 0.65) * 0.10:,.2f} RON")
print(f"Cost total angajator : {cost_angajator:,.2f} RON")

## Pasul 3 — Cum să modifici valorile implicite

Valorile implicite (default) ale calculatorului se află în dicționarul `input_defaults` din `app.py`, la **linia ~408**.

### 3.1 — Modificare salariu și dividende dorite

In [ ]:
# ============================================================
# EDITEAZĂ VALORILE DE MAI JOS conform situației tale:
# ============================================================

# --- OWNER ---
NET_SALARIU   = 6500.0   # RON net/lună dorit de owner
NET_DIVIDENDE = 4000.0   # RON dividende nete/lună dorite

# --- ANGAJAȚI ---
NR_ANGAJATI        = 1       # câți angajați (fără owner)
SALARIU_ANGAJAT_1  = 4325.0  # salariu brut angajat 1 (salariu minim = 4325)

# --- COSTURI FIXE LUNARE (RON) ---
CHIRIE_TEREN     = 2000.0
CHIRIE_CONTAINER = 1500.0
CONTABILITATE    = 500.0
UTILITATI        = 700.0
COMBUSTIBIL      = 800.0
LEASING          = 1500.0
ALTE_COSTURI     = 1000.0

# --- BUSINESS ---
MARJA_PROCENT   = 25     # % marja comerciala (ex: 25 = 25%)
ZILE_LUCRATOARE = 30     # zile/luna
VALOARE_COS     = 12.0   # RON valoare medie cos/client
INCLUDE_CASS    = True   # aplica CASS pe dividende

# --- PREȚURI PRODUSE (RON) ---
PRET_APA      = 3.59
PRET_SUC      = 6.19
PRET_DULCIURI = 5.49
PRET_TIGARI   = 30.0

# --- MIX VÂNZĂRI (trebuie să fie 100%) ---
MIX_APA      = 35.0
MIX_SUC      = 25.0
MIX_DULCIURI = 30.0
MIX_TIGARI   = 10.0

assert MIX_APA + MIX_SUC + MIX_DULCIURI + MIX_TIGARI == 100, \
    f"❌ Suma mix = {MIX_APA+MIX_SUC+MIX_DULCIURI+MIX_TIGARI}% — trebuie să fie 100%!"
print("✅ Valorile sunt valide. Mix total:", MIX_APA+MIX_SUC+MIX_DULCIURI+MIX_TIGARI, "%")

### 3.2 — Calculul complet în notebook (fără Streamlit)

Poți rula calculele direct în Python pentru a verifica rapid rezulttele, fără să pornești aplicația.

In [ ]:
import pandas as pd

# ---- Funcții de calcul (identice cu app.py) ----

def gross_from_net(net):
    return net / 0.585

def employer_cost(gross):
    return gross * (1 + CONFIG['CAM_ANGAJATOR'])

def dividend_cass(net_div_lunar, include_cass):
    if not include_cass or net_div_lunar <= 0:
        return {'plafon': 0, 'cass_anual': 0, 'cass_lunar': 0}
    anual = net_div_lunar * 12
    if anual <= CONFIG['CASS_PLAFON_6SAL']:
        plafon = CONFIG['CASS_PLAFON_6SAL']
    elif anual <= CONFIG['CASS_PLAFON_12SAL']:
        plafon = CONFIG['CASS_PLAFON_12SAL']
    else:
        plafon = CONFIG['CASS_PLAFON_24SAL']
    cass_anual = plafon * CONFIG['COTA_CASS_DIVIDENDE']
    return {'plafon': plafon, 'cass_anual': cass_anual, 'cass_lunar': cass_anual / 12}

def gross_dividends_needed(net_div_lunar, include_cass):
    ci = dividend_cass(net_div_lunar, include_cass)
    brut = (net_div_lunar + ci['cass_lunar']) / (1 - CONFIG['COTA_IMPOZIT_DIVIDENDE'])
    return {'brut_lunar': brut, 'cass_lunar': ci['cass_lunar'],
            'impozit': brut * CONFIG['COTA_IMPOZIT_DIVIDENDE'], 'cass_info': ci}

def pretax_for_dividends(div_brut_lunar):
    return div_brut_lunar / (1 - CONFIG['COTA_IMPOZIT_PROFIT'])

# ---- Calcule ----

gross_owner = gross_from_net(NET_SALARIU)
cost_owner  = employer_cost(gross_owner)

cost_angajati = employer_cost(SALARIU_ANGAJAT_1) * NR_ANGAJATI

costuri_fixe = (CHIRIE_TEREN + CHIRIE_CONTAINER + CONTABILITATE +
                UTILITATI + COMBUSTIBIL + LEASING + ALTE_COSTURI)

div_info    = gross_dividends_needed(NET_DIVIDENDE, INCLUDE_CASS)
profit_nec  = pretax_for_dividends(div_info['brut_lunar'])

total_costuri   = cost_owner + cost_angajati + costuri_fixe + profit_nec
venituri_lunare = total_costuri / (MARJA_PROCENT / 100)
venituri_zilnice = venituri_lunare / ZILE_LUCRATOARE
clienti_zi       = venituri_zilnice / VALOARE_COS

# ---- Afișare rezultate ----

print("=" * 55)
print("         REZULTATE CALCULATOR AFACERE")
print("=" * 55)
print(f"  Cost owner (angajator)     : {cost_owner:>10,.2f} RON/lună")
print(f"  Cost angajați              : {cost_angajati:>10,.2f} RON/lună")
print(f"  Costuri fixe totale        : {costuri_fixe:>10,.2f} RON/lună")
print(f"  Profit pre-tax dividende   : {profit_nec:>10,.2f} RON/lună")
print("-" * 55)
print(f"  TOTAL COSTURI              : {total_costuri:>10,.2f} RON/lună")
print("=" * 55)
print(f"  Marja aplicată             : {MARJA_PROCENT:.0f}%")
print(f"  VENITURI LUNARE NECESARE   : {venituri_lunare:>10,.2f} RON/lună")
print(f"  VENITURI ZILNICE NECESARE  : {venituri_zilnice:>10,.2f} RON/zi")
print(f"  CLIENȚI ESTIMAȚI / ZI      : {clienti_zi:>10.0f} clienți")
print("=" * 55)

### 3.3 — Tabel sensibilitate marja

Verifică cum se schimbă veniturile necesare la diferite marje comerciale.

In [ ]:
marje = [15, 18, 20, 22, 25, 28, 30, 35]

rows = []
for m in marje:
    rev_lunar  = total_costuri / (m / 100)
    rev_zilnic = rev_lunar / ZILE_LUCRATOARE
    clienti    = rev_zilnic / VALOARE_COS
    rows.append({
        'Marja (%)': f"{m}%",
        'Venituri lunare (RON)': f"{rev_lunar:,.0f}",
        'Venituri zilnice (RON)': f"{rev_zilnic:,.0f}",
        'Clienți/zi': f"{clienti:.0f}",
    })

df = pd.DataFrame(rows)
df

### 3.4 — Cum să editezi `input_defaults` în app.py

Dacă vrei că valorile tale să fie preîncărcate la fiecare pornire a aplicației, editează dicționarul `input_defaults` din `app.py`.

**Localizare:** linia ~408 din `app.py`

```python
input_defaults = {
    'moneda': 'RON',            # ← 'RON' sau 'EUR'
    'curs': 5.10,               # ← cursul EUR/RON
    'net_salariu': 6500.0,      # ← salariul net dorit (RON)
    'net_dividende': 4000.0,    # ← dividendele nete dorite (RON/lună)
    'nr_angajati': 1,           # ← câți angajați (fără owner)
    'salariu_angajat_1': 4325.0,# ← brut angajat 1
    'chirie_teren': 2000.0,
    'chirie_container': 1500.0,
    'contabilitate': 500.0,
    'utilitati': 700.0,
    'combustibil': 800.0,
    'leasing': 1500.0,
    'alte_costuri': 1000.0,
    'margin_procent': 25,       # ← IMPORTANT: număr întreg, nu 25.0!
    'include_cass': True,       # ← True/False
    'zile_lucratoare': 30,
    'valoare_cos': 12.0,
    'pret_apa': 3.59,
    'pret_suc': 6.19,
    'pret_dulciuri': 5.49,
    'pret_tigari': 30.0,
    'mix_apa': 35.0,
    'mix_suc': 25.0,
    'mix_dulciuri': 30.0,       # ← suma mix_* trebuie = 100.0
    'mix_tigari': 10.0,
}
```

> ⚠️ **Atenție:** `margin_procent` trebuie să fie un număr **întreg** (`25`, nu `25.0`) pentru că este legat de un slider Streamlit cu step=1.

## Pasul 4 — Pornirea aplicației Streamlit

### Metoda 1 — Din terminal (recomandat)

Deschide un terminal în folderul proiectului și rulează:

```bash
streamlit run app.py
```

Aplicația se va deschide automat în browser la adresa:
- **Local:** http://localhost:8501

### Metoda 2 — Din notebook (celula de mai jos)

> Rulează celula, apoi deschide http://localhost:8501 în browser.

In [ ]:
import subprocess
import time
import webbrowser

print("▶ Pornire aplicatie Streamlit...")
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.headless', 'true'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# Asteapta ca serverul sa porneasca
for _ in range(20):
    line = proc.stdout.readline()
    if 'Local URL' in line or 'localhost' in line:
        print('✅ Aplicatia ruleaza la:', line.strip())
        break
    time.sleep(0.3)

webbrowser.open('http://localhost:8501')
print("\n🌐 Browserul ar trebui sa se deschida automat.")
print("   Daca nu, mergi manual la: http://localhost:8501")
print("\n   Pentru a opri aplicatia, opreste kernel-ul sau apasa Ctrl+C in terminal.")

## Pasul 5 — Ghid interfață Streamlit

### Sidebar (stânga)

| Element | Descriere |
|---|---|
| **Limbă RO/EN** | Schimbă limba interfeței |
| **Monedă RON/EUR** | Afișează valorile în RON sau EUR |
| **Curs EUR/RON** | Apare doar dacă selectezi EUR |
| **CASS dividende** | Toggle pentru includerea CASS pe dividende |
| **Reset** | Resetează toți parametrii la valorile implicite |
| **Încarcă exemplu** | Încarcă scenariul default |

### Secțiuni principale

| Secțiune | Ce editezi |
|---|---|
| 👤 Owner | Salariul net și dividendele dorite |
| 👥 Angajați | Numărul și salariile angajaților |
| 🏠 Costuri fixe | Toate cheltuielile fixe lunare |
| 📊 Parametri business | Marja, zile lucrătoare, valoare coș |
| 🛒 Produse | Prețuri unitare și mix vânzări (%) |

### Rezultate

- **4 carduri** — venituri lunare, zilnice, clienți/zi, produse/zi
- **Breakdownuri detaliate** — salariu owner, angajați, costuri fixe, dividende
- **Tabele comparative** — sensibilitate margine, scenarii dividende, costuri fixe
- **Exports CSV** — descarca raportul curent sau compara versiunile salvate

### Salvare versiuni

1. Setezi toți parametrii dorilți
2. Scrii un nume pentru versiune (ex: „Scenariul optimist
,
3
,
4
,
5

: 
,
: 
6
,
: {},
: [
6
,
,

: 
,
: null,
: 
700000000
,
: {},
: [],
: [
print(f"Valoare medie coș:         {valoare_cos_ron:.2f} RON/client")
print(f"Clienți necesari/zi:       {venituri_zilnice / valoare_cos_ron:.0f} clienți")
print()
print("--- Mix vânzări ---")
for prod, pct, pret in [
    ('Apă',      MIX_APA,      PRET_APA),
    ('Suc',      MIX_SUC,      PRET_SUC),
    ('Dulciuri', MIX_DULCIURI, PRET_DULCIURI),
    ('Țigări',   MIX_TIGARI,   PRET_TIGARI),
]:
    venit_zi = venituri_zilnice * pct / 100
    unit_zi  = venit_zi / pret
    print(f"  {prod:<12} {pct:>4.0f}%  →  {venit_zi:>8,.0f} RON/zi  ({unit_zi:.0f} buc/zi)")

### Scenariu B — Îmi schimb marja și vad impactul

In [ ]:
# Scenariu B: impact schimbare marja
# Modifică valoarea de mai jos și rulează

marja_noua = 30  # % — schimba la valoarea dorita

rev_nou    = total_costuri / (marja_noua / 100)
rev_zi_nou = rev_nou / ZILE_LUCRATOARE

print(f"Marja curentă ({MARJA_PROCENT}%): {venituri_lunare:,.0f} RON/lună → {venituri_zilnice:,.0f} RON/zi")
print(f"Marja nouă    ({marja_noua}%): {rev_nou:,.0f} RON/lună → {rev_zi_nou:,.0f} RON/zi")
print()
delta = venituri_lunare - rev_nou
print(f"Diferență: {delta:+,.0f} RON/lună ({'mai puțin' if delta > 0 else 'mai mult'} venituri necesare)")

### Scenariu C — Angajez un al doilea angajat, cum cresc costurile?

In [ ]:
# Scenariu C: impact angajat suplimentar

salariu_angajat_2 = 4325.0  # RON brut — modifică dacă vrei salariu mai mare

cost_angajat_2 = employer_cost(salariu_angajat_2)
total_cu_angajat_2 = total_costuri + cost_angajat_2
rev_cu_ang2    = total_cu_angajat_2 / (MARJA_PROCENT / 100)
rev_zi_cu_ang2 = rev_cu_ang2 / ZILE_LUCRATOARE

print(f"Cost angajat 2 (angajator): {cost_angajat_2:,.2f} RON/lună")
print()
print(f"Fără angajat 2: {venituri_lunare:>10,.0f} RON/lună necesari ({venituri_zilnice:,.0f}/zi)")
print(f"Cu angajat 2  : {rev_cu_ang2:>10,.0f} RON/lună necesari ({rev_zi_cu_ang2:,.0f}/zi)")
print(f"Diferență     : {rev_cu_ang2 - venituri_lunare:>+10,.0f} RON/lună în plus")

## Pasul 7 — Structura fișierelor proiectului

```
calculator_afacere_v1/
│
├── app.py              ← Aplicația Streamlit principală
├── requirements.txt    ← Dependențe Python
├── history.json        ← Creat automat la prima salvare de versiune
└── ghid_utilizare.ipynb ← Acest notebook
```

### Cum pornești aplicația rapid (Windows)

1. Click dreapta în folderul `calculator_afacere_v1`
2. **„Open in Terminal
,
3